In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ==========================================
# 1. CREACIÓN DE DIMENSIONES (dim_pozos y dim_equipos)
# ==========================================
np.random.seed(42) # Para reproducibilidad

dim_pozos = pd.DataFrame({
    'pozo_id': ['PZ-001', 'PZ-002', 'PZ-003', 'PZ-004', 'PZ-005'],
    'bloque_operativo': ['Llanos - Paz de Ariporo', 'Llanos - Paz de Ariporo', 'Valle Medio del Magdalena', 'Cuenca Putumayo', 'Valle Medio del Magdalena'],
    'tipo_perforacion': ['Direccional', 'Horizontal', 'Vertical', 'Direccional', 'Vertical'],
    'profundidad_objetivo_ft': [12000, 15500, 9500, 11000, 10500]
})

equipos_base = [
    ('Zaranda Primaria', 'Derrick Hyperpool'), ('Zaranda Secundaria', 'Swaco MONGOOSE'),
    ('Zaranda Primaria', 'NOV Brandt KING COBRA'), ('Zaranda Secundaria', 'Derrick Hyperpool'),
    ('Desarenador', 'Nov Brandt 2 conos'), ('Desarenador', 'Derrick 3 conos'),
    ('Deslimador', 'Nov Brandt 16 conos'), ('Deslimador', 'Derrick 20 conos'),
    ('Centrífuga Decantadora', 'Derrick 7200'), ('Centrífuga Decantadora', 'Derrick DE-1000')
]

equipos_data = []
for i in range(1, 26):
    tipo, modelo = equipos_base[i % len(equipos_base)]
    equipos_data.append({
        'equipo_id': f'EQ-{i:03d}',
        'pozo_id': np.random.choice(dim_pozos['pozo_id']),
        'tipo_equipo': tipo,
        'modelo': modelo,
        'estado_inicial': 'Operativo'
    })
dim_equipos = pd.DataFrame(equipos_data)

In [ ]:
# ==========================================
# 2. CREACIÓN DE HECHOS: PARÁMETROS OPERATIVOS
# ==========================================
# Simulamos 3 meses de operación (Mayo, Junio, Julio 2026), tomando un registro cada 6 horas por equipo
fecha_inicio = datetime(2026, 5, 1)
fechas = [fecha_inicio + timedelta(hours=6*i) for i in range(368)] 

parametros_data = []
for equipo in dim_equipos['equipo_id']:
    tipo_eq = dim_equipos.loc[dim_equipos['equipo_id'] == equipo, 'tipo_equipo'].values[0]
    pozo = dim_equipos.loc[dim_equipos['equipo_id'] == equipo, 'pozo_id'].values[0]
    
    for fecha in fechas:
        tipo_lodo = np.random.choice(['WBM', 'OBM'], p=[0.4, 0.6])
        caudal = np.random.uniform(400, 800)
        viscosidad = np.random.uniform(45, 75)
        densidad = np.random.uniform(9.0, 14.5)
        
        # Solo las zarandas usan mallas API
        usa_malla = 'Zaranda' in tipo_eq
        malla = np.random.choice([100, 140, 170, 200, 230]) if usa_malla else 0
        
        parametros_data.append({
            'equipo_id': equipo,
            'pozo_id': pozo,
            'fecha_hora': fecha,
            'tipo_lodo': tipo_lodo,
            'caudal_gpm': round(caudal, 1),
            'viscosidad_funnel': round(viscosidad, 1),
            'densidad_lpg': round(densidad, 1),
            'malla_api': malla
        })

fact_parametros = pd.DataFrame(parametros_data)

In [ ]:
# ==========================================
# 3. CREACIÓN DE HECHOS: FALLAS NPT (Regla de Negocio)
# ==========================================
fallas_data = []
falla_id_counter = 1

for index, row in fact_parametros.iterrows():
    probabilidad_falla = 0.01 # Probabilidad base del 1%
    
    # REGLA DE NEGOCIO: Lodo OBM + Alta Viscosidad (>60) + Malla Fina (>=170) + Alto Caudal (>600)
    if row['tipo_lodo'] == 'OBM' and row['viscosidad_funnel'] > 60 and row['malla_api'] >= 170 and row['caudal_gpm'] > 600:
        probabilidad_falla = 0.65 # Sube a 65% de probabilidad de falla
        
    if np.random.rand() < probabilidad_falla:
        tipo_falla = np.random.choice(['Rotura de Malla', 'Desbordamiento de Fluido', 'Falla Motor Vibrador'])
        horas_npt = round(np.random.uniform(1.5, 6.0), 1)
        
        fallas_data.append({
            'falla_id': f'F-{falla_id_counter:04d}',
            'equipo_id': row['equipo_id'],
            'fecha_hora': row['fecha_hora'],
            'tipo_falla': tipo_falla,
            'horas_npt': horas_npt
        })
        falla_id_counter += 1

fact_fallas = pd.DataFrame(fallas_data)

In [ ]:
# ==========================================
# 4. EXPORTAR A CSV
# ==========================================
dim_pozos.to_csv('dim_pozos.csv', index=False)
dim_equipos.to_csv('dim_equipos.csv', index=False)
fact_parametros.to_csv('fact_parametros_operativos.csv', index=False)
fact_fallas.to_csv('fact_npt_fallas.csv', index=False)

print(f"Archivos generados exitosamente:")
print(f"- dim_pozos: {len(dim_pozos)} registros")
print(f"- dim_equipos: {len(dim_equipos)} registros")
print(f"- fact_parametros: {len(fact_parametros)} registros")
print(f"- fact_fallas: {len(fact_fallas)} registros de NPT simulados")